# Entropía e Información: por que -log(p) mide el error
**Objetivo**: entender por que la función de pérdida Cross-Entropy usa logaritmos,
que mide exactamente, y como la teoría de la información de Shannon (1948) le da
un significado preciso más allá de 'penaliza errores'.

**Estructura:**
1. ¿Que es información? Bits como unidad de sorpresa
2. La función de auto-información: I(x) = -log2(P(x))
3. Por que el logaritmo: la única función que hace sumar la información
4. Entropía de Shannon: H(X) = -Sum p*log2(p)
5. Cross-entropy: medir el coste de estar equivocado
6. Conexión con el entrenamiento de redes neuronales
7. Ejercicio técnico y ejercicio de decisión

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import pathlib

IMG = pathlib.Path('images')
IMG.mkdir(exist_ok=True)

np.random.seed(42)
print('Entorno listo')

Entorno listo


---

## 1. ¿Que es 'información'?

Claude Shannon (1948) propuso una definición matemática de información que no tiene nada
que ver con el significado semántico del mensaje. Su pregunta era:

> ¿Cuanta información porta un mensaje si ya sabía (con cierta probabilidad) lo que iba a decir?

La respuesta: **la información de un evento es inversamente proporcional a su probabilidad**.
Un evento seguro (P=1) no aporta nada nuevo. Un evento rarísimo (P~0) aporta muchísimo.

**La unidad: el bit**

Si un evento tiene probabilidad 1/2, su información es exactamente **1 bit**.
Esto tiene sentido: para saber si salió cara o cruz en una moneda justa necesitas
exactamente 1 bit (0 o 1). Esa es la cantidad mínima de información para describir
el resultado.

```
moneda justa:  P(cara) = 1/2  ->  I = -log2(1/2) = 1 bit
dado de 6:     P(3)    = 1/6  ->  I = -log2(1/6) ~= 2.58 bits
carta (52):    P(As)   = 1/52 ->  I = -log2(1/52) ~= 5.70 bits
```

Cada vez que el espacio de posibilidades se duplica, necesitas **1 bit más** para describir el resultado.

In [2]:
import numpy as np

eventos = [
    ('Moneda (cara/cruz)', 0.5),
    ('Dado de 6 caras', 1/6),
    ('Carta en baraja de 52', 1/52),
    ('Letra en espanol (~27)', 1/27),
    ('Respuesta binaria si/no', 0.5),
]

nombres = [e[0] for e in eventos]
probs   = [e[1] for e in eventos]
bits    = [-np.log2(p) for p in probs]

print('Tabla de bits por evento:')
print(f'  {"Evento":<30} {"P(x)":>8} {"Bits":>8}')
print('  ' + '-'*50)
for n, p, b in zip(nombres, probs, bits):
    print(f'  {n:<30} {p:>8.4f} {b:>8.4f}')

Tabla de bits por evento:
  Evento                             P(x)     Bits
  --------------------------------------------------
  Moneda (cara/cruz)               0.5000   1.0000
  Dado de 6 caras                  0.1667   2.5850
  Carta en baraja de 52            0.0192   5.7004
  Letra en espanol (~27)           0.0370   4.7549
  Respuesta binaria si/no          0.5000   1.0000


In [3]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eventos = [
    ('Moneda (cara/cruz)', 0.5),
    ('Dado de 6 caras', 1/6),
    ('Carta en baraja de 52', 1/52),
    ('Letra en espanol (~27)', 1/27),
    ('Respuesta binaria si/no', 0.5),
]
nombres = [e[0] for e in eventos]
probs   = [e[1] for e in eventos]
bits    = [-np.log2(p) for p in probs]

colores = ['#1565C0', '#E65100', '#2E7D32', '#6A1B9A', '#00838F']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(nombres, bits, color=colores, height=0.55, edgecolor='white')
for bar, b, p in zip(bars, bits, probs):
    ax.text(b + 0.05, bar.get_y() + bar.get_height()/2,
            f'{b:.2f} bits  (P={p:.4f})',
            va='center', ha='left', fontsize=9.5, color='#333')
ax.set_xlabel('Bits de informacion (auto-informacion)', fontsize=11)
ax.set_title('I(x) = -log2(P(x)): bits necesarios para describir cada evento',
             fontsize=11, pad=10)
ax.set_xlim(0, 8)
ax.axvline(1, color='#888', linestyle='--', linewidth=1, alpha=0.6)
ax.text(1.05, -0.6, '1 bit (moneda justa)', fontsize=8, color='#888', ha='left')
ax.grid(axis='x', alpha=0.25)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('images/B04A_fig01.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B04A_fig01.png')

Guardado: images/B04A_fig01.png


---

## 2. La función de auto-información: I(x) = -log2(P(x))

La definición formal de la **auto-información** (o *self-information*, también llamada *surprisal*):

```
I(x) = -log2(P(x))
```

Propiedades:
- P(x) = 1   ->  I(x) = 0      (certeza: ninguna sorpresa, cero información)
- P(x) = 0.5 ->  I(x) = 1      (moneda justa: 1 bit)
- P(x) -> 0  ->  I(x) -> +inf  (evento rarísimo: información enorme)
- I(x) >= 0 siempre (un evento nunca aporta información negativa)

> **Antes de seguir:** ¿Que evento cotidiano de tu trabajo aporta más 'información'
> según esta definición - un retraso de entrega en un proveedor fiable (P~0.05)
> o un retraso de un proveedor habitualmente impuntual (P~0.70)?

<details>
<summary>Orientación para el instructor</summary>

Una respuesta madura menciona: el retraso del proveedor fiable porta ~4.3 bits (es sorprendente),
el del impuntual porta ~0.51 bits (es esperado). La 'información' no mide importancia sino novedad.
Si nadie responde, preguntar: ¿cual de los dos retrasos te haría llamar inmediatamente al responsable?
Señal de comprensión: el alumno distingue entre 'información' en el sentido de Shannon (novedad/sorpresa)
y 'importancia' en el sentido de negocio.

</details>

In [4]:
import numpy as np

casos = [
    ('Proveedor fiable retrasa (raro)',     0.05),
    ('Proveedor impuntual retrasa (normal)', 0.70),
    ('Evento certero',                      1.00),
    ('Evento imposible (limite)',            0.001),
]

print('Auto-informacion I(x) = -log2(P(x)):')
print(f'  {"Caso":<42} {"P(x)":>6} {"I(x) bits":>10}')
print('  ' + '-'*62)
for nombre, p in casos:
    bits = -np.log2(p)
    print(f'  {nombre:<42} {p:>6.3f} {bits:>10.4f}')
print()
print('Proveedor fiable: I ~= 4.32 bits  (sorprendente, mucha informacion nueva)')
print('Proveedor impuntual: I ~= 0.51 bits (esperado, poca informacion nueva)')

Auto-informacion I(x) = -log2(P(x)):
  Caso                                         P(x)  I(x) bits
  --------------------------------------------------------------
  Proveedor fiable retrasa (raro)             0.050     4.3219
  Proveedor impuntual retrasa (normal)        0.700     0.5146
  Evento certero                              1.000    -0.0000
  Evento imposible (limite)                   0.001     9.9658

Proveedor fiable: I ~= 4.32 bits  (sorprendente, mucha informacion nueva)
Proveedor impuntual: I ~= 0.51 bits (esperado, poca informacion nueva)


In [5]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

p = np.linspace(0.001, 1.0, 1000)
I = -np.log2(p)

puntos = [
    (0.5,    1.0,              'Moneda (0.5 -> 1 bit)',    '#1565C0'),
    (1/6,    -np.log2(1/6),   'Dado (1/6 -> 2.58 bits)', '#E65100'),
    (1/52,   -np.log2(1/52),  'Carta (1/52 -> 5.70 bits)','#2E7D32'),
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(p, I, color='#1565C0', linewidth=2.5, label='I(x) = -log2(P(x))')

for px, iy, etiqueta, color in puntos:
    ax.scatter([px], [iy], s=90, color=color, zorder=5)
    ax.annotate(etiqueta,
                xy=(px, iy), xytext=(px + 0.06, iy + 0.35),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

ax.axhline(0, color='#ccc', linewidth=0.8)
ax.set_xlabel('Probabilidad P(x)', fontsize=11)
ax.set_ylabel('Auto-informacion I(x) en bits', fontsize=11)
ax.set_title('I(x) = -log2(P(x)): la sorpresa aumenta con la rareza', fontsize=11, pad=10)
ax.set_xlim(0, 1.0)
ax.set_ylim(-0.2, 10)
ax.grid(alpha=0.25)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('images/B04A_fig02.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B04A_fig02.png')

Guardado: images/B04A_fig02.png


---

## 3. Por que el logaritmo: la propiedad de aditividad

El logaritmo no es una elección arbitraria. Es la **única función** que cumple esta
propiedad deseable:

> **Si dos eventos son independientes, su información total debe ser la suma de sus informaciones individuales.**

Ejemplo: lanzar dos monedas independientes.
- I(cara en moneda 1) = 1 bit
- I(cara en moneda 2) = 1 bit
- I(cara en ambas) = ? -> debe ser 2 bits

Con la definición I(x) = -log2(P(x)):
```
I(cara y cara) = -log2(P(cara) * P(cara)) = -log2(1/4) = 2 bits
```

La propiedad del logaritmo: log(a*b) = log(a) + log(b).
Probabilidades independientes se **multiplican** -> logaritmo las convierte en **suma** de información.

Ninguna otra función hace eso manteniendo la monotonía (más raro = más información).

In [6]:
import numpy as np

print('Verificacion de aditividad del logaritmo:')
print()

# Lanzar dos monedas
p_cara = 0.5
I_moneda1 = -np.log2(p_cara)
I_moneda2 = -np.log2(p_cara)
I_conjunta_log = -np.log2(p_cara * p_cara)
I_conjunta_suma = I_moneda1 + I_moneda2

print('Lanzar dos monedas justas:')
print(f'  I(cara moneda 1) = -log2(0.5) = {I_moneda1:.2f} bit')
print(f'  I(cara moneda 2) = -log2(0.5) = {I_moneda2:.2f} bit')
print(f'  I(cara y cara)   = -log2(0.25) = {I_conjunta_log:.2f} bits   [via probabilidad conjunta]')
print(f'  I1 + I2          = {I_conjunta_suma:.2f} bits   [via suma directa]')
print(f'  Son iguales: {np.isclose(I_conjunta_log, I_conjunta_suma)}')
print()

# Dado + carta
p_dado = 1/6
p_carta = 1/52
print('Lanzar dado (P=1/6) y sacar carta (P=1/52):')
print(f'  I(dado=3) = {-np.log2(p_dado):.4f} bits')
print(f'  I(carta)  = {-np.log2(p_carta):.4f} bits')
print(f'  I(dado=3 y carta) = {-np.log2(p_dado * p_carta):.4f} bits')
print(f'  Suma individual   = {-np.log2(p_dado) + -np.log2(p_carta):.4f} bits')
print(f'  Son iguales: {np.isclose(-np.log2(p_dado * p_carta), -np.log2(p_dado) + -np.log2(p_carta))}')

Verificacion de aditividad del logaritmo:

Lanzar dos monedas justas:
  I(cara moneda 1) = -log2(0.5) = 1.00 bit
  I(cara moneda 2) = -log2(0.5) = 1.00 bit
  I(cara y cara)   = -log2(0.25) = 2.00 bits   [via probabilidad conjunta]
  I1 + I2          = 2.00 bits   [via suma directa]
  Son iguales: True

Lanzar dado (P=1/6) y sacar carta (P=1/52):
  I(dado=3) = 2.5850 bits
  I(carta)  = 5.7004 bits
  I(dado=3 y carta) = 8.2854 bits
  Suma individual   = 8.2854 bits
  Son iguales: True


---

## 4. Entropía de Shannon: H(X) = -Sum p*log2(p)

La **entropía** es el valor esperado de la auto-información:

```
H(X) = E[I(X)] = -Sum P(x) * log2(P(x))
```

Es la **información media** que obtienes por mensaje, o equivalentemente, la
**incertidumbre media** del sistema antes de recibir el mensaje.

**Interpretación como desorden:**
- H = 0: certeza total (una sola clase con P=1). Sin información nueva posible.
- H máxima: distribución uniforme (todos los resultados igualmente probables). Máxima incertidumbre.

Ejemplos:
- Moneda justa (P=0.5/0.5): H = 1 bit - máxima incertidumbre para 2 clases
- Moneda trucada (P=0.9/0.1): H ~= 0.47 bits - poca incertidumbre
- Dado justo (6 caras equiprobables): H = log2(6) ~= 2.58 bits

In [7]:
import numpy as np

def entropia(dist):
    dist = np.array(dist, dtype=float)
    dist = dist[dist > 0]
    return float(-np.sum(dist * np.log2(dist)))

distribuciones = [
    ('Moneda justa',          [0.5,  0.5]),
    ('Moneda trucada',        [0.9,  0.1]),
    ('Dado justo (6)',        [1/6]*6),
    ('Certeza total',         [1.0,  0.0]),
    ('Uniforme (4 clases)',   [0.25]*4),
]

print('Entropia de Shannon H(X):')
print(f'  {"Distribucion":<28} {"H (bits)":>10}')
print('  ' + '-'*42)
for nombre, dist in distribuciones:
    h = entropia(dist)
    print(f'  {nombre:<28} {h:>10.4f}')
print()
print('A mayor incertidumbre, mayor entropia.')
print('H maxima para N clases: log2(N)  -->  uniforme(6)=2.585, uniforme(4)=2.0')

Entropia de Shannon H(X):
  Distribucion                   H (bits)
  ------------------------------------------
  Moneda justa                     1.0000
  Moneda trucada                   0.4690
  Dado justo (6)                   2.5850
  Certeza total                   -0.0000
  Uniforme (4 clases)              2.0000

A mayor incertidumbre, mayor entropia.
H maxima para N clases: log2(N)  -->  uniforme(6)=2.585, uniforme(4)=2.0


In [8]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def entropia(dist):
    dist = np.array(dist, dtype=float)
    dist = dist[dist > 0]
    return float(-np.sum(dist * np.log2(dist)))

# Parte 1: curva H(p, 1-p) binaria
ps = np.linspace(0.01, 0.99, 200)
hs = [-p*np.log2(p) - (1-p)*np.log2(1-p) for p in ps]

# Parte 2: 4 distribuciones de 6 clases
dists = {
    'Uniforme (6 clases)':  [1/6]*6,
    'Sesgada':              [0.70, 0.10, 0.05, 0.05, 0.05, 0.05],
    'Casi-cierta':          [0.95, 0.01, 0.01, 0.01, 0.01, 0.01],
    'Bimodal':              [0.45, 0.45, 0.025, 0.025, 0.025, 0.025],
}
colores_dist = ['#1565C0', '#E65100', '#2E7D32', '#6A1B9A']

fig = plt.figure(figsize=(14, 9))
gs = GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.45)

ax_curva = fig.add_subplot(gs[0, :])
ax_curva.plot(ps, hs, color='#1565C0', linewidth=2.5)
ax_curva.axvline(0.5, color='#E65100', linestyle='--', linewidth=1.5,
                 label='Maximo en p=0.5 (H=1 bit)')
ax_curva.scatter([0.5], [1.0], s=100, color='#E65100', zorder=5)
h_trucada = -0.9*np.log2(0.9) - 0.1*np.log2(0.1)
ax_curva.scatter([0.9], [h_trucada], s=80, color='#2E7D32', zorder=5,
                 label=f'Moneda trucada p=0.9 (H~={h_trucada:.2f} bits)')
ax_curva.set_xlabel('Probabilidad p de la clase 1', fontsize=10)
ax_curva.set_ylabel('H(p, 1-p) en bits', fontsize=10)
ax_curva.set_title('Entropia binaria: maxima incertidumbre cuando todo es posible',
                   fontsize=11, pad=8)
ax_curva.legend(fontsize=9)
ax_curva.grid(alpha=0.25)

for idx, (nombre, dist) in enumerate(dists.items()):
    ax = fig.add_subplot(gs[1, idx])
    h_val = entropia(dist)
    clases = [f'C{i+1}' for i in range(len(dist))]
    ax.bar(clases, dist, color=colores_dist[idx], alpha=0.85, edgecolor='white')
    ax.set_title(f'{nombre}\nH = {h_val:.3f} bits', fontsize=9, pad=6)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('P(x)', fontsize=8)
    ax.tick_params(labelsize=8)
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Entropia de Shannon: incertidumbre segun la distribucion de probabilidad',
             fontsize=12, y=1.01)
plt.savefig('images/B04A_fig03.png', dpi=120, bbox_inches='tight')
plt.close()
print('Entropias calculadas:')
for nombre, dist in dists.items():
    print(f'  {nombre:<30} H = {entropia(dist):.4f} bits')
print('Guardado: images/B04A_fig03.png')

Entropias calculadas:
  Uniforme (6 clases)            H = 2.5850 bits
  Sesgada                        H = 1.5568 bits
  Casi-cierta                    H = 0.4025 bits
  Bimodal                        H = 1.5690 bits
Guardado: images/B04A_fig03.png


---

## 5. Cross-entropy: el coste de estar equivocado

La **cross-entropy** H(P, Q) mide cuantos bits necesitas por mensaje si usas
el código de Q para transmitir mensajes que en realidad siguen la distribución P:

```
H(P, Q) = -Sum P(x) * log2(Q(x))
```

- P = distribución real (los datos verdaderos)
- Q = distribución predicha (las probabilidades del modelo)

**Cuando Q = P exactamente:** H(P, P) = H(P) -> mínimo posible, igual a la entropía real.
**Cuando Q != P:** H(P, Q) > H(P) -> bits extra por estar equivocado.

La diferencia H(P, Q) - H(P) se llama **divergencia KL (Kullback-Leibler)**:
mide exactamente 'cuantos bits extra paga el modelo por equivocarse'.

**Conexión con el entrenamiento:**
En clasificación multiclase, P es la distribución one-hot (P=1 en la clase correcta, 0 en el resto):

```
H(P_one-hot, Q_modelo) = -log(Q[clase_correcta])
```

Esto es exactamente la fórmula `L = -log(y_pred[clase_correcta])` de la sección 4.1.
La cross-entropy no es una fórmula elegida por conveniencia: es la medida óptima de
'cuantos bits extra usa el modelo por no conocer la verdadera distribución de los datos'.

In [9]:
import numpy as np

def cross_entropy(P, Q):
    P = np.array(P, dtype=float)
    Q = np.clip(np.array(Q, dtype=float), 1e-12, 1.0)
    return float(-np.sum(P * np.log2(Q)))

def entropia(P):
    P = np.array(P, dtype=float)
    P = P[P > 0]
    return float(-np.sum(P * np.log2(P)))

# P real: clase 'alto' con certeza (one-hot)
P_real = [0.0, 0.0, 1.0]  # clases: bajo, medio, alto

modelos = [
    ('Perfecto',         [0.001, 0.001, 0.998]),
    ('Bueno',            [0.10,  0.15,  0.75]),
    ('Mediocre',         [0.20,  0.30,  0.50]),
    ('Marginal',         [0.33,  0.33,  0.34]),
    ('Equivocado',       [0.70,  0.20,  0.10]),
]

H_P = entropia(P_real)
print(f'H(P_real) = {H_P:.4f} bits (entropia de one-hot = 0)')
print()
print(f'  {"Modelo":<15} {"CE H(P,Q)":>12} {"KL = CE - H":>12}')
print('  ' + '-'*42)
for nombre, Q in modelos:
    ce = cross_entropy(P_real, Q)
    kl = ce - H_P
    print(f'  {nombre:<15} {ce:>12.4f} {kl:>12.4f}')
print()
print('El modelo perfecto: CE = 0, sin bits extra.')
print('El modelo equivocado: paga muchos bits extra (KL alto).')

H(P_real) = -0.0000 bits (entropia de one-hot = 0)

  Modelo             CE H(P,Q)  KL = CE - H
  ------------------------------------------
  Perfecto              0.0029       0.0029
  Bueno                 0.4150       0.4150
  Mediocre              1.0000       1.0000
  Marginal              1.5564       1.5564
  Equivocado            3.3219       3.3219

El modelo perfecto: CE = 0, sin bits extra.
El modelo equivocado: paga muchos bits extra (KL alto).


In [10]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

q = np.linspace(0.01, 0.999, 1000)
ce = -np.log(q)  # P = one-hot clase correcta = 1

puntos_marcados = [
    (0.999, -np.log(0.999), 'Q~1.0  -> CE~0.00 (perfecto)',  '#2E7D32'),
    (0.5,   -np.log(0.5),   'Q=0.5  -> CE~=0.69',            '#1565C0'),
    (0.1,   -np.log(0.1),   'Q=0.1  -> CE~=2.30',            '#E65100'),
    (0.01,  -np.log(0.01),  'Q=0.01 -> CE~=4.61',            '#6A1B9A'),
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(q, ce, color='#1565C0', linewidth=2.5, label='CE = -log(Q) cuando P es one-hot')
ax.fill_between(q, ce, where=(q < 0.5), alpha=0.12, color='#E65100',
                label='Zona de alta penalizacion (Q < 0.5)')

for qx, cy, etiqueta, color in puntos_marcados:
    ax.scatter([qx], [cy], s=80, color=color, zorder=5)
    offset_x = -0.28 if qx > 0.8 else 0.04
    offset_y = 0.25
    ax.annotate(etiqueta, xy=(qx, cy),
                xytext=(qx + offset_x, cy + offset_y),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

ax.set_xlabel('Probabilidad asignada por el modelo Q a la clase correcta', fontsize=11)
ax.set_ylabel('Cross-entropy = -log(Q)', fontsize=11)
ax.set_title('H(P,Q) = -log(Q): cuanto paga el modelo por equivocarse', fontsize=11, pad=10)
ax.set_xlim(0, 1.05)
ax.set_ylim(-0.1, 6)
ax.axvline(0.5, color='#aaa', linestyle='--', linewidth=1, alpha=0.7)
ax.grid(alpha=0.25)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('images/B04A_fig04.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B04A_fig04.png')

Guardado: images/B04A_fig04.png


---

## 6. Resumen: de Shannon a la función de pérdida

```
Shannon (1948):      I(x) = -log2(P(x))       -> bits de información de un evento
Entropía:            H(X) = -Sum p*log2(p)     -> información media del sistema
Cross-entropy:       H(P,Q) = -Sum P*log(Q)    -> bits usando modelo Q para realidad P
Loss en red:         L = -log(y_pred[clase])   -> cross-entropy con P = one-hot

El entrenamiento minimiza H(P,Q):
  -> Q se acerca a P
  -> El modelo aprende la distribución real de los datos
  -> Los bits extra (divergencia KL) tienden a cero
```

**¿Por que no MSE para clasificación?**

El error cuadrático medio mide distancia geométrica entre vectores de probabilidad.
La cross-entropy mide bits de información. Para distribuciones de probabilidad,
los bits son la métrica natural: dos predicciones [0.51, 0.49] y [0.99, 0.01]
tienen MSE similar si se comparan con [1, 0], pero cross-entropy las distingue
radicalmente - la segunda usa el código correcto, la primera no.

In [11]:
import numpy as np

# Predicciones (prob asignada a la clase correcta)
confianzas = [0.99, 0.80, 0.60, 0.51, 0.10]
labels_reales = [1, 1, 1, 1, 0]  # todos correctos excepto el ultimo

print('Comparativa Cross-Entropy vs MSE:')
print(f'  {"Confianza":>10} {"CrossEntropy":>14} {"MSE":>10} {"Acierto":>8}')
print('  ' + '-'*48)
for conf, y_real in zip(confianzas, labels_reales):
    ce_val  = -np.log(np.clip(conf, 1e-7, 1))
    mse_val = (conf - 1)**2 + (1 - conf)**2
    acierto = 'SI' if (conf >= 0.5 and y_real == 1) or (conf < 0.5 and y_real == 0) else 'NO'
    print(f'  {conf:>10.2f} {ce_val:>14.4f} {mse_val:>10.4f} {acierto:>8}')
print()
print('CE distingue claramente 0.99 de 0.80 de 0.51.')
print('MSE apenas los diferencia porque la distancia geometrica es pequena.')
print('Ambos aciertan con threshold=0.5, pero CE sabe cual modelo es mas confiado.')

Comparativa Cross-Entropy vs MSE:
   Confianza   CrossEntropy        MSE  Acierto
  ------------------------------------------------
        0.99         0.0101     0.0002       SI
        0.80         0.2231     0.0800       SI
        0.60         0.5108     0.3200       SI
        0.51         0.6733     0.4802       SI
        0.10         2.3026     1.6200       SI

CE distingue claramente 0.99 de 0.80 de 0.51.
MSE apenas los diferencia porque la distancia geometrica es pequena.
Ambos aciertan con threshold=0.5, pero CE sabe cual modelo es mas confiado.


---

## 7. Ejercicio técnico

### Enunciado

Tienes un modelo de clasificación de 3 clases (bajo/medio/alto riesgo de churn).
Para 4 clientes, el modelo predice estas probabilidades y las clases reales son:

| Cliente | Pred. bajo | Pred. medio | Pred. alto | Clase real |
|---------|-----------|-------------|------------|------------|
| A       | 0.70      | 0.20        | 0.10       | bajo       |
| B       | 0.20      | 0.60        | 0.20       | medio      |
| C       | 0.10      | 0.30        | 0.60       | alto       |
| D       | 0.40      | 0.35        | 0.25       | alto       |

**Pregunta:** Calcula la cross-entropy loss para cada cliente y la loss media del batch.
¿Qué cliente genera más pérdida? ¿Por que el cliente D tiene alta pérdida a pesar de que
su predicción no es completamente incorrecta?

### TODO: tu solución aquí

In [12]:
import numpy as np

# Completa las siguientes lineas
y_pred = np.array([
    [0.70, 0.20, 0.10],  # Cliente A
    [0.20, 0.60, 0.20],  # Cliente B
    [0.10, 0.30, 0.60],  # Cliente C
    [0.40, 0.35, 0.25],  # Cliente D
])
y_true = np.array([0, 1, 2, 2])  # bajo, medio, alto, alto

# TODO: calcula la prob de la clase correcta para cada cliente
correct_probs = None  # y_pred[np.arange(len(y_pred)), y_true]

# TODO: calcula la cross-entropy loss para cada cliente
losses = None  # -np.log(correct_probs)

print('Tu solucion:')
print(f'  correct_probs = {correct_probs}')
print(f'  losses = {losses}')

Tu solucion:
  correct_probs = None
  losses = None


In [13]:
# Solucion - ejecutar para ver resultado
import numpy as np

y_pred = np.array([
    [0.70, 0.20, 0.10],  # Cliente A
    [0.20, 0.60, 0.20],  # Cliente B
    [0.10, 0.30, 0.60],  # Cliente C
    [0.40, 0.35, 0.25],  # Cliente D
])
y_true = np.array([0, 1, 2, 2])  # bajo, medio, alto, alto

y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
correct_probs = y_pred_clipped[np.arange(len(y_pred)), y_true]
losses = -np.log(correct_probs)

for i, (loss, prob) in enumerate(zip(losses, correct_probs)):
    cliente = ['A', 'B', 'C', 'D'][i]
    print(f'Cliente {cliente}: prob_correcta={prob:.2f}, loss={loss:.4f}, bits={loss/np.log(2):.4f}')

print(f'\nLoss media del batch: {losses.mean():.4f}')
print(f'\nCliente D: aunque predice 0.25 en alto (correcto), es muy baja confianza -> loss alta')
print(f'La cross-entropy penaliza mas a D que a B (0.60 en correcto) porque B sabe mas')

Cliente A: prob_correcta=0.70, loss=0.3567, bits=0.5146
Cliente B: prob_correcta=0.60, loss=0.5108, bits=0.7370
Cliente C: prob_correcta=0.60, loss=0.5108, bits=0.7370
Cliente D: prob_correcta=0.25, loss=1.3863, bits=2.0000

Loss media del batch: 0.6912

Cliente D: aunque predice 0.25 en alto (correcto), es muy baja confianza -> loss alta
La cross-entropy penaliza mas a D que a B (0.60 en correcto) porque B sabe mas


**Interpretación del resultado:**

- **Cliente A**: prob=0.70 en la clase correcta (bajo). Loss moderada: el modelo es bastante confiado.
- **Cliente B**: prob=0.60 en la clase correcta (medio). Loss similar a A.
- **Cliente C**: prob=0.60 en la clase correcta (alto). Bien calibrado.
- **Cliente D**: prob=0.25 en la clase correcta (alto). Loss alta porque el modelo asigna
  más probabilidad a clases incorrectas. No es que falle, es que no sabe.

La cross-entropy no solo penaliza los errores: penaliza la **baja confianza** en la respuesta correcta.
Dos modelos que aciertan pueden tener loss muy distinta si uno lo hace con 0.51 y otro con 0.99.

---

## 8. Ejercicio de decisión

### Caso: ¿debería el equipo comercial ver la cross-entropy?

Tu modelo de clasificación de riesgo de churn tiene una accuracy del 87% en producción.
El equipo de negocio quiere un único número para saber 'como va el modelo'.

Alguien propone publicar la cross-entropy en el dashboard junto con la accuracy.
El equipo de negocio pregunta: '¿Que significa que la cross-entropy subió de 0.31 a 0.58
pero la accuracy sigue siendo 87%?'

**Pregunta:** ¿Como explicarías esta situación? ¿Qué información aporta la cross-entropy
que la accuracy no captura? ¿Debería estar ese número en el dashboard de negocio?

**Argumenta tu respuesta** considerando:
- ¿Qué puede estar pasando con la distribución de las predicciones?
- ¿Es esto una señal de alerta temprana o ruido?
- ¿Qué audiencia necesita este dato - técnica, negocio, o ambas?

### Tu razonamiento aquí

In [14]:
import numpy as np

np.random.seed(42)

# Simulamos dos momentos del modelo
# Momento 1: modelo confiado, accuracy 87%%
# Momento 2: mismo accuracy, pero menos confiado (distributional shift)

def simular_predicciones(n=1000, confianza_alta=True):
    y_true = (np.random.rand(n) < 0.13).astype(int)  # 13%% churn
    probs = []
    for y in y_true:
        if confianza_alta:
            # Modelo confiado: predicciones cerca de 0 o 1
            p = np.random.beta(8, 2) if y == 1 else np.random.beta(2, 8)
        else:
            # Modelo menos confiado: predicciones mas centradas (drift)
            p = np.random.beta(2, 2) if y == 1 else np.random.beta(2, 2)
            p = 0.4 + 0.2 * p  # comprimir entre 0.4 y 0.6
        probs.append(p)
    probs = np.clip(np.array(probs), 1e-7, 1 - 1e-7)
    y_pred_bin = (probs >= 0.5).astype(int)
    accuracy = np.mean(y_pred_bin == y_true)
    ce = float(-np.mean(
        y_true * np.log(probs) + (1 - y_true) * np.log(1 - probs)
    ))
    return accuracy, ce

acc1, ce1 = simular_predicciones(confianza_alta=True)
acc2, ce2 = simular_predicciones(confianza_alta=False)

print('Simulacion de distributional shift:')
print(f'  {"":20} {"Accuracy":>10} {"Cross-Entropy":>14}')
print('  ' + '-'*48)
print(f'  {"Modelo confiado (T1)":<20} {acc1:>10.3f} {ce1:>14.4f}')
print(f'  {"Modelo con drift (T2)":<20} {acc2:>10.3f} {ce2:>14.4f}')
print()
print('Observacion:')
print('  Accuracy casi igual. Cross-entropy sube notablemente.')
print('  El modelo acierta pero cada vez con menos confianza.')
print('  Senal de alerta temprana antes de que accuracy caiga.')

Simulacion de distributional shift:
                         Accuracy  Cross-Entropy
  ------------------------------------------------
  Modelo confiado (T1)      0.988         0.2417
  Modelo con drift (T2)      0.542         0.6902

Observacion:
  Accuracy casi igual. Cross-entropy sube notablemente.
  El modelo acierta pero cada vez con menos confianza.
  Senal de alerta temprana antes de que accuracy caiga.


### Criterios de evaluación

- **Identifica** que cross-entropy subiendo con accuracy estable = modelo menos confiado
  (predicciones más cercanas a 0.5, no a 0 o 1)
- **Reconoce** que es señal de alerta de distributional shift o degradación gradual
  antes de que accuracy caiga
- **Distingue** que métricas son para la audiencia técnica vs de negocio:
 - Negocio: accuracy, recall en churn, coste estimado de errores
 - Técnica: cross-entropy, calibración, drift de distribución de scores
- **Concluye** que la cross-entropy no debe ir al dashboard de negocio sin traducción

---

## Recapitulación

| Concepto | Fórmula | Significado |
|----------|---------|-------------|
| Auto-información | I(x) = -log2(P(x)) | Bits de sorpresa de un evento |
| Entropía | H(X) = -Sum p*log2(p) | Incertidumbre media del sistema |
| Cross-entropy | H(P,Q) = -Sum P*log(Q) | Coste de usar modelo Q para realidad P |
| Loss clasificación | L = -log(y_pred[clase]) | CE con P one-hot |
| Divergencia KL | KL = H(P,Q) - H(P) | Bits extra por equivocarse |

**Cadena lógica:**
Shannon define información como sorpresa.
La sorpresa de eventos independientes se suma.
Solo el logaritmo convierte multiplicación de probabilidades en suma de información.
La entropía promedia la sorpresa.
La cross-entropy mide el precio de usar un modelo equivocado.
Minimizar la loss es minimizar los bits extra que paga el modelo.

## Conexión con el entrenamiento de la red

Cuando se aplica backpropagation con cross-entropy loss:

1. El gradiente de `-log(y_pred[clase])` respecto a y_pred es `-1/y_pred[clase]`
2. Para probabilidades bajas (modelo muy equivocado), el gradiente es grande
3. Para probabilidades altas (modelo confiado), el gradiente es pequeño

Esto crea una actualización automáticamente proporcional al error:
cuanto más equivocado está el modelo, más fuerte es el ajuste de pesos.

El MSE no tiene esta propiedad para clasificación: su gradiente puede ser
muy pequeño incluso cuando la predicción es totalmente errónea (problema de
'vanishing gradient' en la capa de salida).

In [15]:
import numpy as np

print('Gradiente de -log(p) respecto a p:')
print('  d(-log(p))/dp = -1/p')
print()

confianzas = [0.99, 0.80, 0.50, 0.20, 0.05]
print(f'  {"Prob. correcta":>16} {"Loss CE":>10} {"Gradiente -1/p":>16}')
print('  ' + '-'*46)
for p in confianzas:
    loss = -np.log(p)
    grad = -1 / p
    print(f'  {p:>16.2f} {loss:>10.4f} {grad:>16.4f}')
print()
print('Cuando p=0.05 (muy equivocado): gradiente = -20 -> actualizacion fuerte')
print('Cuando p=0.99 (muy seguro):     gradiente = -1.01 -> ajuste minimo')
print()
print('La cross-entropy implementa automaticamente: error grande -> correccion grande.')

Gradiente de -log(p) respecto a p:
  d(-log(p))/dp = -1/p

    Prob. correcta    Loss CE   Gradiente -1/p
  ----------------------------------------------
              0.99     0.0101          -1.0101
              0.80     0.2231          -1.2500
              0.50     0.6931          -2.0000
              0.20     1.6094          -5.0000
              0.05     2.9957         -20.0000

Cuando p=0.05 (muy equivocado): gradiente = -20 -> actualizacion fuerte
Cuando p=0.99 (muy seguro):     gradiente = -1.01 -> ajuste minimo

La cross-entropy implementa automaticamente: error grande -> correccion grande.


---

## Assets generados

- `images/B04A_fig01.png` - Tabla de bits por evento (moneda, dado, carta, letra)
- `images/B04A_fig02.png` - Curva I(x) = -log2(P(x)): la sorpresa aumenta con la rareza
- `images/B04A_fig03.png` - Entropía de Shannon: distribuciones con distinta incertidumbre
- `images/B04A_fig04.png` - Cross-entropy H(P,Q) como función de la confianza del modelo